# Install & Import

In [ ]:
from snowflake.snowpark.context import get_active_session
import time
import os

session = get_active_session()

requirements = """
rioxarray==0.17.0
pystac==1.11.0
pystac_client==0.9.0
planetary_computer==1.0.0
odc-stac==0.3.10
shapely==2.1.1
rasterio==1.4.3
tqdm==4.66.5
pandas==2.3.0
xarray==2025.3.1
matplotlib==3.10.3
geopandas==1.1.1
seaborn==0.13.2
scikit-learn==1.5.2
netCDF4==1.7.2
adlfs==2025.8.0
zarr==2.17.2
dask==2024.10.0
xarray[complete]
numcodecs==0.12.1
"""


with open('/tmp/requirements.txt', 'w') as f:
    f.write(requirements)

print("File requirements.txt berhasil dibuat di memori lokal (/tmp)")

session.sql(f"""
    PUT file:///tmp/requirements.txt 
    snow://workspace/USER$.PUBLIC.DEFAULT$/versions/live/
    AUTO_COMPRESS=FALSE 
    OVERWRITE=TRUE
""").collect()

print("File ter-upload ke environment Snowflake!")

start_time = time.time()
print("Start Install Package ...")

!pip install uv
!uv pip install -r /tmp/requirements.txt

elapsed_time = time.time() - start_time
print(f"\n✓ Instalasi Package Selesai 100%!")
print(f"  Waktu yang dihabiskan: {elapsed_time:.1f} detik ({elapsed_time/60:.1f} menit)")

In [ ]:
import pandas as pd
import os
from snowflake.snowpark.context import get_active_session
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '../../'))
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Project root is set to: {project_root}")
print(f"Is 'src' in root?: {'src' in os.listdir(project_root)}")

from src.data_ingestion import enrich_dataset_with_external_api

session = get_active_session()
data_stage_path = "@EXTERNAL_DATA_STAGE"

In [ ]:
session.sql("USE DATABASE SNOWFLAKE_LEARNING_DB").collect()
session.sql("USE SCHEMA PUBLIC").collect()

static_files_to_process = [
    "zaf_pop_2025_CN_100m_R2025A_v1.tif",
    "SA_NLC_2022_ALBERS.tif",
    "SA_NLC_2022_ALBERS.tif.vat.dbf",
    "SA_NLC_2020_ALBERS.tif",
    "SA_NLC_2020_ALBERS.tif.vat.dbf",
    "BasinATLAS_v1.parquet",
    "RiverATLAS_v1.parquet"
]

for file_name in static_files_to_process:
    temporary_target_path = f"/tmp/{file_name}"
    if not os.path.exists(temporary_target_path):
        print(f"Downloading {file_name} from Stage to local environment...")
        session.file.get(f"{data_stage_path}/{file_name}", "/tmp/")
        
print("✓ Semua file geospasial statis siap di /tmp/")

In [ ]:
session.file.get(f"{data_stage_path}/water_quality_training_dataset.csv", "/tmp/")
base_water_quality_df = pd.read_csv("/tmp/water_quality_training_dataset.csv")

# Ekstraksi unik Latitude, Longitude, dan Sample Date
extraction_base_df = base_water_quality_df[['Latitude', 'Longitude', 'Sample Date']].drop_duplicates().reset_index(drop=True)

print(f"Total koordinat unik yang akan di-enrich: {len(extraction_base_df)}")

In [ ]:
print("Memulai proses ekstraksi fitur eksternal. Mohon tunggu...")

external_features_df = enrich_dataset_with_external_api(
    dataframe=extraction_base_df,
    latitude_col='Latitude',
    longitude_col='Longitude',
    date_col='Sample Date'
)

display(external_features_df.head())

In [ ]:
key_columns = ['Latitude', 'Longitude', 'Sample Date']
feature_columns = [col for col in external_features_df.columns if col not in key_columns]
final_export_df = external_features_df[key_columns + feature_columns]

# Perubahan menjadi format Parquet
output_file_name = "external_features_training.parquet"
output_local_path = f"/tmp/{output_file_name}"

# Menyimpan dataframe ke format Parquet
final_export_df.to_parquet(output_local_path, index=False, engine='pyarrow', compression='snappy')

# Upload kembali ke Snowflake Stage
session.file.put(
    f"file://{output_local_path}", 
    data_stage_path, 
    auto_compress=False, 
    overwrite=True
)

print(f"✓ Ekstraksi Selesai! Data super ringan berhasil disimpan ke {data_stage_path}/{output_file_name}")